In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 83.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 5.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=65d45ce9c4d8025c66bdf270e88d9dd94a8ec218003cf0b9c66d003b26759dcc
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [ ]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.

In [58]:
# Quantum Number Generator
simulator = BasicSimulator()

def quantum_random_bit() -> int:
    # create a circuit with 1 qubit & 1 classical bit
    qc = QuantumCircuit(1, 1)
    qc.h(0)
    qc.measure(0, 0)
    job = simulator.run(transpile(qc, simulator), shots=1)
    result = job.result()
    counts = result.get_counts()
    return int(list(counts.keys())[0])

def quantum_random_bits(n: int) -> list:
    # generate multiple random bits
    return [quantum_random_bit() for _ in range(n)]

In [59]:
# Qubit Preperation (Alice)
# basis 0 =  standard
# basis 1 = diagonal
def prepare_qubit(bit: int, basis: int) -> QuantumCircuit:
    qc = QuantumCircuit(1, 1)
    # encode bit 1 using X gate
    if bit == 1:
        qc.x(0)
    # change to diagonal basis if needed
    if basis == 1:
        qc.h(0)
    return qc


In [60]:
# Qubit Measurement (Bob)
# if basis = 1, apply H before measuring
def measure(state_qc: QuantumCircuit, basis: int) -> int:
    qc = QuantumCircuit(1, 1)  # fresh circuit with 1 qubit and 1 classical bit
    qc = qc.compose(state_qc)  # apply Alice's preparation gates
    if basis == 1:
        qc.h(0)
    qc.measure(0, 0)
    job = simulator.run(transpile(qc, simulator), shots=1)
    counts = job.result().get_counts()
    return int(list(counts.keys())[0])

In [61]:
# BB84 Simulation (No Attacker)
NUM_QUBITS = 20

# ALICE
# generates random bits and bases
alice_bits = quantum_random_bits(NUM_QUBITS)
alice_bases = quantum_random_bits(NUM_QUBITS)

# prepares qubits
transmitted_qubits = [
    prepare_qubit(alice_bits[i], alice_bases[i])
    for i in range(NUM_QUBITS)
]

# BOB
# randomly chooses bases
bob_bases = quantum_random_bits(NUM_QUBITS)
# measures all received qubits
bob_bits = [
    measure(transmitted_qubits[i], bob_bases[i])
    for i in range(NUM_QUBITS)
]


In [62]:
# Basis Reconciliation
matching_indices = [i for i in range(NUM_QUBITS) if alice_bases[i] == bob_bases[i]]

# build the sifted key
alice_key = [alice_bits[i] for i in matching_indices]
bob_key   = [bob_bits[i]   for i in matching_indices]

In [63]:
print("\nRaw transmission")
print(f"Index:       {list(range(NUM_QUBITS))}")
print(f"Alice bits:  {alice_bits}")
print(f"Alice bases: {alice_bases}  (0=std, 1=diag)")
print(f"Bob bases:   {bob_bases}")
print(f"Bob bits:    {bob_bits}")

print("\nPost-processing")
print(f"Matching positions: {matching_indices}")
print(f"Alice's key: {alice_key}")
print(f"Bob's key:   {bob_key}")
print(f"Keys match:  {alice_key == bob_key}")
print(f"Key length:  {len(alice_key)} bits out of {NUM_QUBITS} transmitted")


Raw transmission
Index:       [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
Alice bits:  [1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0]
Alice bases: [1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1]  (0=std, 1=diag)
Bob bases:   [0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0]
Bob bits:    [0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 1]

Post-processing
Matching positions: [4, 6, 7, 8, 11, 12, 14, 15, 16, 17, 18]
Alice's key: [1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0]
Bob's key:   [1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0]
Keys match:  True
Key length:  11 bits out of 20 transmitted


In [64]:
# Attacker detection setup
SAMPLE_FRACTION = 0.5 # fraction of key used for checking
ATTACK_THRESHOLD = 0.1 # max acceptable error rate

# number of bits to compare
sample_size = max(1, int(len(alice_key) * SAMPLE_FRACTION))
# take 1st part of key as sample
sample_indices = list(range(sample_size))
# count how many bits differ between alice and bob
mismatches = sum(1 for i in sample_indices if alice_key[i] != bob_key[i])
error_rate = mismatches / sample_size # compute error rate in the sample

# Check Results
print(f"Error rate: {error_rate:.2%}")
if error_rate > ATTACK_THRESHOLD:
    print("Attack detected.")
else:
    print("No attack detected")

Error rate: 0.00%
No attack detected
